# Install Libraries/Modules/Packages

In [2]:
#Author: Laura Dominguez
#Created on: 04/11/2026
#Course: BCIS 566 - Business Analytics II
#Semester Project

# Data analysis tools
import pandas as pd
import numpy as np

# For timing models
import time

# Visualization
import matplotlib.pyplot as plt
from matplotlib import cm
import seaborn as sns

# Train and Testing Split
from sklearn.model_selection import train_test_split

# Classification Models
from sklearn.linear_model import Perceptron, LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn import tree

# Classification Metrics
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Regression Models
from sklearn.linear_model import LinearRegression

# Regression Metrics
from sklearn.metrics import r2_score, mean_squared_error

# Dimension Reduction
from sklearn.decomposition import PCA, KernelPCA as KPCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA

# Feature Scaling
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Ensemble Methods
from sklearn.ensemble import RandomForestClassifier, BaggingClassifier, AdaBoostClassifier, VotingClassifier
from xgboost import XGBClassifier

# Model Selection
from sklearn.model_selection import GridSearchCV

# Functions

In [3]:
def print_info(df):
    """Prints number of rows, columns, and missing values in a dataset"""
    print(f"The dataset has:\n{df.shape[0]} rows\n{df.shape[1]} columns")
    print(f"The dataset has a total of {df.isna().sum().sum()} missing values")
    print(f"The missing values per column are:\n{df.isna().sum()}")

def clean_dataset(df):
    """Removes any missing values in a dataset"""
    if df.isna().any().any():
        #remove missing values
        df = df.dropna()

        #recalculate column and row number
        print(f"Number of rows in dataset after removing missing values: {df.shape[0]}")
        print(f"Number of columns in dataset after removing missing values: {df.shape[1]}")
    
    else:
        print("The dataset has no missing values.")
    
    return df

def standardize_data(X_train, X_test):
    """Standerdizes feature X training data and testing"""
    
    # Intialize StandardScaler
    std = StandardScaler()

    # Fit and transform the scaler to X_train and X, test
    X_train_std = std.fit_transform(X_train)
    X_test_std = std.transform(X_test)
    
    return X_train_std, X_test_std

def get_model(name):
    """Returns sci-kit learn model name.Model names need to be inputted with appropriate capitlization"""
    models = {"DecisionTree":DecisionTreeClassifier,
              "LogisticRegression":LogisticRegression,
              "KNN":KNeighborsClassifier,
              "LinearSVM": SVC,
              "NonLinearSVM": SVC,
              "RandomForest": RandomForestClassifier,
              "Bagging": BaggingClassifier,
              "AdaBoost": AdaBoostClassifier,
              "XGBoost": XGBClassifier,
              "MajorityVoting": VotingClassifier}

    # Strip whitespace and remove spaces 
    name = name.strip().replace(" ", "") # clean the string

    if name not in models:
        raise ValueError(f"Invalid classifier name. Please choose from {list(models.keys())}")

    return models[name]

## Classification

In [4]:
def print_class_metrics(y_test,y_pred, y_prob):
    """Prints Metrics of a classification Model.
       Returns confusion matrix, accuracy, precision, recall, and f1_score"""
    # Store metrics
    c_matrix = confusion_matrix(y_test, y_pred)
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average="weighted")
    recall = recall_score(y_test, y_pred, average="weighted")
    f1 = f1_score(y_test, y_pred, average="weighted")
    roc_auc = roc_auc_score(y_test, y_prob, multi_class="ovr", average="weighted")

    class_metrics = {"confusion_matrix": c_matrix, 
                     "accuracy": accuracy,
                     "precision": precision, 
                     "recall": recall, 
                     "f1": f1,
                     "roc_auc": roc_auc}
    
    # Print the misclassified examples and error
    print(f"Misclassifed examples: {(y_test != y_pred).sum()} out of {y_test.shape[0]}")
    print("Error: %.3f" % (((y_test != y_pred).sum())/(y_test.shape[0])))
    
    # Print classifier confusion matrix, accuracy, precision, and recall
    print(f"Confusion_Matrix:\n{c_matrix}")
    print("Accuracy: %.3f" % accuracy)
    print("Precision %.3f" % precision)
    print("Recall: %.3f" % recall)
    print("F1_score: %.3f" % f1)
    print("ROC_AUC score: %.3f" % roc_auc)

    return class_metrics
  
def classification_model(model_name,X_train_std, X_test_std, y_train, y_test, **kwargs):
    """Initalizes, Fits, and Predicts using a Classification Model"""

    #Choose classifier
    clf_class = get_model(model_name)

    # intitialize the model
    model = clf_class(**kwargs)

    # Get start time of training model
    train_start = time.time()

    # Fit the model 
    model.fit(X_train_std, y_train)

    # Stop the timer
    train_time = time.time() - train_start

    # Print training time
    print(f"Training time: {train_time:.4f} seconds")

    # Get start time of testing model
    test_start = time.time()

    # Predict the overall class, ie predicted hand number
    y_pred = model.predict(X_test_std)
    y_prob = model.predict_proba(X_test_std)[:,1]

    # Stop the timer
    test_time = time.time() - test_start

    # Print testing time
    print(f"Testing time: {test_time:.4f} seconds")

    # Print Metrics
    metrics = print_class_metrics(y_test, y_pred, y_prob)

    row = pd.DataFrame({ "Model": [model_name],
                           "Parameters": [kwargs],
                           "Training Time (sec)": [round(train_time, 4)],
                           "Testing Time (sec)": [round(test_time,4)], 
                           "Accuracy" : round(metrics["accuracy"],2),
                           "Precision":round(metrics["precision"],2),
                           "Recall": round(metrics["recall"],2),
                           "F1_score": round(metrics["f1"],2),
                           "ROC-AUC": round(metrics["roc_auc"],2)})
    
    # Return model
    return model, row

## Ensemble

In [25]:
def ensemble_learning(model_name,X_train, X_test, y_train, y_test,**kwargs):
    """Initalizes, Fits, and Predicts using a Ensemble Model"""

    # Choose ensemble
    clf_class = get_model(model_name)

    # Intitialize the model
    model = clf_class(**kwargs)

    # Get start time of training model
    train_start = time.time()

    # Fit the model 
    model.fit(X_train, y_train)

    # Stop the timer
    train_time = time.time() - train_start

    # Print training time
    print(f"Training time: {train_time:.4f} seconds")

    # Get start time of testing model
    test_start = time.time()

    # Predict the overall class, ie predicted hand number
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:,1]

    # Stop the timer
    test_time = time.time() - test_start

    # Print testing time
    print(f"Testing time: {test_time:.4f} seconds")

    # Print Metrics
    metrics = print_class_metrics(y_test, y_pred,y_prob)

    row = pd.DataFrame({ "Model": [model_name],
                           "Parameters": [kwargs],
                           "Training Time (sec)": [round(train_time, 4)],
                           "Testing Time (sec)": [round(test_time,4)], 
                           "Accuracy" : round(metrics["accuracy"],2),
                           "Precision":round(metrics["precision"],2),
                           "Recall": round(metrics["recall"],2),
                           "F1_score": round(metrics["f1"],2),
                           "ROC-AUC": round(metrics["roc_auc"],2)})

    # Return model
    return model, y_pred, row

def grid_search(model_name, X_train, y_train, param_grid, cv=5, scoring="accuracy", model_params=None):

    if model_params is None:
        model_params = {}

    clf_class = get_model(model_name)

    # Bagging special case
    if model_name == "Bagging":
        model = clf_class(
            estimator=model_params.get("estimator", ),
            random_state=42
        )

    # Voting special case
    elif model_name == "MajorityVoting":
        model = clf_class(
            estimators=model_params.get("estimators", []),
            voting=model_params.get("voting", "hard")
        )

    else:
        model = clf_class(**model_params)

    grid = GridSearchCV(model, param_grid=param_grid, cv=cv, scoring=scoring)
    grid.fit(X_train, y_train)

    results = pd.DataFrame(grid.cv_results_)
    results_sorted = results.sort_values(by="rank_test_score")

    print(results_sorted[["params", "mean_test_score", "rank_test_score"]].head(5))

    return grid


def evaluate_model(model, X_train, X_test, y_train, y_test):
    """Used with grid_search() to evalaute the best estimator"""

    # Start training time
    start = time.time()

    # Fit best estimator model
    model.fit(X_train, y_train)

    # End training time 
    train_time = time.time() - start

    # Start Time
    start = time.time()

    # Predict the overall class, ie predicted hand number
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:,1]

    # Stop Timer
    test_time = time.time() - start

    print(f"Prediction time: {test_time:.6f} seconds")

     # Print Metrics
    metrics = print_class_metrics(y_test, y_pred, y_prob)

    row = pd.DataFrame({ "Model": [type(model).__name__],
                           "Parameters": [model.get_params()],
                           "Training Time (sec)": [round(train_time, 4)],
                           "Testing Time (sec)": [round(test_time,4)], 
                           "Accuracy" : round(metrics["accuracy"],2),
                           "Precision":round(metrics["precision"],2),
                           "Recall": round(metrics["recall"],2),
                           "F1_score": round(metrics["f1"],2),
                           "ROC-AUC": round(metrics["roc_auc"],2)})

    return row

# Load Dataset

In [6]:
train_census_incomeDF = pd.read_csv("DATA/adult.data", header=None)

col_names = ["age","workclass", "fnlwgt","education","education_num","marital_status", "occupation", "relationship","race", "sex", 
             "capital_gain", "capital_loss", "hours_per_week", "native_country", "income_label"]

train_census_incomeDF.columns = col_names

train_census_incomeDF

,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income_label
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32556,27,Private,257302,Assoc-acdm,12,Married-civ-spouse,Tech-support,Wife,White,Female,0,0,38,United-States,<=50K
32557,40,Private,154374,HS-grad,9,Married-civ-spouse,Machine-op-inspct,Husband,White,Male,0,0,40,United-States,>50K
32558,58,Private,151910,HS-grad,9,Widowed,Adm-clerical,Unmarried,White,Female,0,0,40,United-States,<=50K
32559,22,Private,201490,HS-grad,9,Never-married,Adm-clerical,Own-child,White,Male,0,0,20,United-States,<=50K


In [7]:
test_census_incomeDF = pd.read_csv("DATA/adult.test", header=None)

col_names = ["age","workclass", "fnlwgt","education","education_num","marital_status", "occupation", "relationship","race", "sex", 
             "capital_gain", "capital_loss", "hours_per_week", "native_country", "income_label"]

test_census_incomeDF.columns = col_names

test_census_incomeDF

,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income_label
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,<=50K.
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,<=50K.
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,>50K.
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,>50K.
4,18,?,103497,Some-college,10,Never-married,?,Own-child,White,Female,0,0,30,United-States,<=50K.
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16276,39,Private,215419,Bachelors,13,Divorced,Prof-specialty,Not-in-family,White,Female,0,0,36,United-States,<=50K.
16277,64,?,321403,HS-grad,9,Widowed,?,Other-relative,Black,Male,0,0,40,United-States,<=50K.
16278,38,Private,374983,Bachelors,13,Married-civ-spouse,Prof-specialty,Husband,White,Male,0,0,50,United-States,<=50K.
16279,44,Private,83891,Bachelors,13,Divorced,Adm-clerical,Own-child,Asian-Pac-Islander,Male,5455,0,40,United-States,<=50K.


In [8]:
# combine datasets since I want to control the train/test split later
census_incomeDF = pd.concat([train_census_incomeDF, test_census_incomeDF], ignore_index=True)
census_incomeDF

,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income_label
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48837,39,Private,215419,Bachelors,13,Divorced,Prof-specialty,Not-in-family,White,Female,0,0,36,United-States,<=50K.
48838,64,?,321403,HS-grad,9,Widowed,?,Other-relative,Black,Male,0,0,40,United-States,<=50K.
48839,38,Private,374983,Bachelors,13,Married-civ-spouse,Prof-specialty,Husband,White,Male,0,0,50,United-States,<=50K.
48840,44,Private,83891,Bachelors,13,Divorced,Adm-clerical,Own-child,Asian-Pac-Islander,Male,5455,0,40,United-States,<=50K.


In [9]:
# Print information regarding dataset
print_info(census_incomeDF)

# Clean strings in columns to avoid regex 
census_incomeDF = census_incomeDF.apply(lambda col: col.str.strip() if col.dtype == "object" else col)

# Convert "?" to numpy Nan
census_incomeDF  = census_incomeDF.replace("?", np.nan)

# Print info again
print_info(census_incomeDF)

# remove missing values
census_incomeDF = clean_dataset(train_census_incomeDF)

# print info again
print_info(census_incomeDF)


The dataset has:
48842 rows
15 columns
The dataset has a total of 0 missing values
The missing values per column are:
age               0
workclass         0
fnlwgt            0
education         0
education_num     0
marital_status    0
occupation        0
relationship      0
race              0
sex               0
capital_gain      0
capital_loss      0
hours_per_week    0
native_country    0
income_label      0
dtype: int64
The dataset has:
48842 rows
15 columns
The dataset has a total of 6465 missing values
The missing values per column are:
age                  0
workclass         2799
fnlwgt               0
education            0
education_num        0
marital_status       0
occupation        2809
relationship         0
race                 0
sex                  0
capital_gain         0
capital_loss         0
hours_per_week       0
native_country     857
income_label         0
dtype: int64
The dataset has no missing values.
The dataset has:
32561 rows
15 columns
The dataset has 

In [10]:
# Establish X and Y in training set
X = census_incomeDF.iloc[:,:14]
y = census_incomeDF["income_label"]

# Encode categorical columns in X before splitting
X = pd.get_dummies(X, drop_first=True)

# Train test split data
X_train, X_test, y_train, y_test = train_test_split(X,y,train_size=0.8,stratify=y,random_state=42)

# Standerdize train and test X
X_train_std, X_test_std = standardize_data(X_train, X_test)

# Classification

In [11]:
col_names = ["Model","Parameters","Training Time (sec)", "Testing Time (sec)", 
             "Accuracy","Precision", "Recall", "F1_score", "ROC-AUC"]
class_results_table = pd.DataFrame(columns=col_names)

## DecisionTree

In [12]:
# Choose the model
class_model= "Decision Tree"

# Without Dimension Reduction but with Feature Scaling, perform decision tree classification on dataset
dt, row = classification_model(class_model,X_train_std, X_test_std, 
                               y_train, y_test,criterion="gini", max_depth=2, random_state=42)

# Add metrics to results table 
class_results_table = pd.concat([class_results_table, row], ignore_index=True)

# print results table 
class_results_table

Training time: 0.0808 seconds
Testing time: 0.0339 seconds
Misclassifed examples: 1113 out of 6513
Error: 0.171
Confusion_Matrix:
[[4696  249]
 [ 864  704]]
Accuracy: 0.829
Precision 0.819
Recall: 0.829
F1_score: 0.813
ROC_AUC score: 0.835


C:\Users\laura\AppData\Local\Temp\ipykernel_7112\1553325746.py:9: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  class_results_table = pd.concat([class_results_table, row], ignore_index=True)


,Model,Parameters,Training Time (sec),Testing Time (sec),Accuracy,Precision,Recall,F1_score,ROC-AUC
0,Decision Tree,"{'criterion': 'gini', 'max_depth': 2, 'random_...",0.0808,0.0339,0.83,0.82,0.83,0.81,0.84


In [13]:
# Select classification Model
class_model = DecisionTreeClassifier(random_state=42)

# Set a list of parameters and # of folds
param_grid = {"max_depth":[None,2,5,10]}
cv = 5

# Call the grid search function
grid_tree  = GridSearchCV(class_model, param_grid=param_grid, cv=cv, scoring="accuracy")
grid_tree.fit(X_train_std, y_train)

results = pd.DataFrame(grid_tree.cv_results_)
results_sorted = results.sort_values(by="rank_test_score")

print(results_sorted[["params", "mean_test_score", "rank_test_score"]].head(5))

# Grab the best model
best_dt = grid_tree.best_estimator_

# Print the best parameters of the best estimator
print("Best Parameters:")
print(grid_tree.best_params_)

# Evaluate the model
row = evaluate_model(best_dt, X_train_std, X_test_std, y_train, y_test)

# Add metrics to results table 
class_results_table = pd.concat([class_results_table, row], ignore_index=True)

# print results table 
class_results_table


                params  mean_test_score  rank_test_score
3    {'max_depth': 10}         0.853616                1
2     {'max_depth': 5}         0.846975                2
1     {'max_depth': 2}         0.827895                3
0  {'max_depth': None}         0.812999                4
Best Parameters:
{'max_depth': 10}
Prediction time: 0.003962 seconds
Misclassifed examples: 938 out of 6513
Error: 0.144
Confusion_Matrix:
[[4530  415]
 [ 523 1045]]
Accuracy: 0.856
Precision 0.853
Recall: 0.856
F1_score: 0.854
ROC_AUC score: 0.902


,Model,Parameters,Training Time (sec),Testing Time (sec),Accuracy,Precision,Recall,F1_score,ROC-AUC
0,Decision Tree,"{'criterion': 'gini', 'max_depth': 2, 'random_...",0.0808,0.0339,0.83,0.82,0.83,0.81,0.84
1,DecisionTreeClassifier,"{'ccp_alpha': 0.0, 'class_weight': None, 'crit...",0.2613,0.0040,0.86,0.85,0.86,0.85,0.90


## Logistic Regression

In [14]:
# Choose the model
class_model= "Logistic Regression"

# Without Dimension Reduction but with Feature Scaling, perform decision tree classification on dataset
log, row = classification_model(class_model,X_train_std, X_test_std, 
                                y_train, y_test,random_state=42, 
                                penalty='l2', # regularization type
                                C=0.5,        # regularization strength
                                solver='lbfgs',  # optimizer
                                max_iter=200)    

# Add metrics to results table 
class_results_table = pd.concat([class_results_table, row], ignore_index=True)

# print results table 
class_results_table

Training time: 0.3025 seconds
Testing time: 0.0020 seconds
Misclassifed examples: 944 out of 6513
Error: 0.145
Confusion_Matrix:
[[4598  347]
 [ 597  971]]
Accuracy: 0.855
Precision 0.849
Recall: 0.855
F1_score: 0.851
ROC_AUC score: 0.909


,Model,Parameters,Training Time (sec),Testing Time (sec),Accuracy,Precision,Recall,F1_score,ROC-AUC
0,Decision Tree,"{'criterion': 'gini', 'max_depth': 2, 'random_...",0.0808,0.0339,0.83,0.82,0.83,0.81,0.84
1,DecisionTreeClassifier,"{'ccp_alpha': 0.0, 'class_weight': None, 'crit...",0.2613,0.0040,0.86,0.85,0.86,0.85,0.90
2,Logistic Regression,"{'random_state': 42, 'penalty': 'l2', 'C': 0.5...",0.3025,0.0020,0.86,0.85,0.86,0.85,0.91


In [15]:
# Select classification Model
log_model = LogisticRegression(max_iter=200, random_state=42)

param_grid = {
    "C": [0.01, 0.1, 0.5, 1, 10, 100],
    "penalty": ["l2"]  # required for l1
}

cv = 5

grid_log = GridSearchCV(
    log_model,
    param_grid=param_grid,
    cv=cv,
    scoring="accuracy"
)

grid_log.fit(X_train_std, y_train)

results = pd.DataFrame(grid_log.cv_results_)
results_sorted = results.sort_values(by="rank_test_score")

print(results_sorted[["params", "mean_test_score", "rank_test_score"]].head(5))

# Grab the best model
best_log = grid_log.best_estimator_

# Print the best parameters of the best estimator
print("Best Parameters:")
print(grid_log.best_params_)

# Evaluate the model
row = evaluate_model(best_log, X_train_std, X_test_std, y_train, y_test)

# Add metrics to results table 
class_results_table = pd.concat([class_results_table, row], ignore_index=True)

# print results table 
class_results_table


                        params  mean_test_score  rank_test_score
3    {'C': 1, 'penalty': 'l2'}         0.850315                1
4   {'C': 10, 'penalty': 'l2'}         0.850276                2
2  {'C': 0.5, 'penalty': 'l2'}         0.850276                3
5  {'C': 100, 'penalty': 'l2'}         0.850238                4
1  {'C': 0.1, 'penalty': 'l2'}         0.850161                5
Best Parameters:
{'C': 1, 'penalty': 'l2'}
Prediction time: 0.002497 seconds
Misclassifed examples: 944 out of 6513
Error: 0.145
Confusion_Matrix:
[[4598  347]
 [ 597  971]]
Accuracy: 0.855
Precision 0.849
Recall: 0.855
F1_score: 0.851
ROC_AUC score: 0.909


,Model,Parameters,Training Time (sec),Testing Time (sec),Accuracy,Precision,Recall,F1_score,ROC-AUC
0,Decision Tree,"{'criterion': 'gini', 'max_depth': 2, 'random_...",0.0808,0.0339,0.83,0.82,0.83,0.81,0.84
1,DecisionTreeClassifier,"{'ccp_alpha': 0.0, 'class_weight': None, 'crit...",0.2613,0.0040,0.86,0.85,0.86,0.85,0.90
2,Logistic Regression,"{'random_state': 42, 'penalty': 'l2', 'C': 0.5...",0.3025,0.0020,0.86,0.85,0.86,0.85,0.91
3,LogisticRegression,"{'C': 1, 'class_weight': None, 'dual': False, ...",0.2813,0.0025,0.86,0.85,0.86,0.85,0.91


## KNN

In [16]:
# Choose the model
class_model= "KNN"

# Without Dimension Reduction but with Feature Scaling, perform decision tree classification on dataset
knn, row = classification_model(class_model,X_train_std, X_test_std, 
                                y_train, y_test, n_neighbors=2, metric="euclidean")

# Add metrics to results table 
class_results_table = pd.concat([class_results_table, row], ignore_index=True)

# print results table 
class_results_table

Training time: 0.1197 seconds
Testing time: 3.1863 seconds
Misclassifed examples: 1260 out of 6513
Error: 0.193
Confusion_Matrix:
[[4665  280]
 [ 980  588]]
Accuracy: 0.807
Precision 0.791
Recall: 0.807
F1_score: 0.785
ROC_AUC score: 0.785


,Model,Parameters,Training Time (sec),Testing Time (sec),Accuracy,Precision,Recall,F1_score,ROC-AUC
0,Decision Tree,"{'criterion': 'gini', 'max_depth': 2, 'random_...",0.0808,0.0339,0.83,0.82,0.83,0.81,0.84
1,DecisionTreeClassifier,"{'ccp_alpha': 0.0, 'class_weight': None, 'crit...",0.2613,0.0040,0.86,0.85,0.86,0.85,0.90
2,Logistic Regression,"{'random_state': 42, 'penalty': 'l2', 'C': 0.5...",0.3025,0.0020,0.86,0.85,0.86,0.85,0.91
3,LogisticRegression,"{'C': 1, 'class_weight': None, 'dual': False, ...",0.2813,0.0025,0.86,0.85,0.86,0.85,0.91
4,KNN,"{'n_neighbors': 2, 'metric': 'euclidean'}",0.1197,3.1863,0.81,0.79,0.81,0.79,0.79


In [19]:
# Select classification Model
knn_model = KNeighborsClassifier()

param_grid = {
    "n_neighbors": [2,4,6,8],
    "metric": ["euclidean","manhattan"]
}

cv = 5

grid_knn = GridSearchCV(
    knn_model,
    param_grid=param_grid,
    cv=cv,
    scoring="accuracy"
)

grid_knn.fit(X_train_std, y_train)

results = pd.DataFrame(grid_knn.cv_results_)
results_sorted = results.sort_values(by="rank_test_score")

print(results_sorted[["params", "mean_test_score", "rank_test_score"]].head(5))

# Grab the best model
best_knn = grid_knn.best_estimator_

# Print the best parameters of the best estimator
print("Best Parameters:")
print(grid_knn.best_params_)

# Evaluate the model
row = evaluate_model(best_knn, X_train_std, X_test_std, y_train, y_test)

# Add metrics to results table 
class_results_table = pd.concat([class_results_table, row], ignore_index=True)

# print results table 
class_results_table


c:\Users\laura\miniconda3\Lib\site-packages\sklearn\model_selection\_validation.py:953: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "c:\Users\laura\miniconda3\Lib\site-packages\sklearn\model_selection\_validation.py", line 942, in _score
    scores = scorer(estimator, X_test, y_test, **score_params)
  File "c:\Users\laura\miniconda3\Lib\site-packages\sklearn\metrics\_scorer.py", line 308, in __call__
    return self._score(partial(_cached_call, None), estimator, X, y_true, **_kwargs)
           ~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\laura\miniconda3\Lib\site-packages\sklearn\metrics\_scorer.py", line 400, in _score
    y_pred = method_caller(
        estimator,
    ...<2 lines>...
        pos_label=pos_label,
    )
  File "c:\Users\laura\miniconda3\Lib\site-packages\sklearn\metrics\_scorer.py", line 90, in _cached_call


                                      params  mean_test_score  rank_test_score
3  {'metric': 'euclidean', 'n_neighbors': 8}         0.822213                1
2  {'metric': 'euclidean', 'n_neighbors': 6}         0.820255                2
1  {'metric': 'euclidean', 'n_neighbors': 4}         0.817529                3
0  {'metric': 'euclidean', 'n_neighbors': 2}         0.807355                4
4  {'metric': 'manhattan', 'n_neighbors': 2}              NaN                5
Best Parameters:
{'metric': 'euclidean', 'n_neighbors': 8}
Prediction time: 0.927788 seconds
Misclassifed examples: 1097 out of 6513
Error: 0.168
Confusion_Matrix:
[[4617  328]
 [ 769  799]]
Accuracy: 0.832
Precision 0.822
Recall: 0.832
F1_score: 0.821
ROC_AUC score: 0.861


,Model,Parameters,Training Time (sec),Testing Time (sec),Accuracy,Precision,Recall,F1_score,ROC-AUC
0,Decision Tree,"{'criterion': 'gini', 'max_depth': 2, 'random_...",0.0808,0.0339,0.83,0.82,0.83,0.81,0.84
1,DecisionTreeClassifier,"{'ccp_alpha': 0.0, 'class_weight': None, 'crit...",0.2613,0.0040,0.86,0.85,0.86,0.85,0.90
2,Logistic Regression,"{'random_state': 42, 'penalty': 'l2', 'C': 0.5...",0.3025,0.0020,0.86,0.85,0.86,0.85,0.91
3,LogisticRegression,"{'C': 1, 'class_weight': None, 'dual': False, ...",0.2813,0.0025,0.86,0.85,0.86,0.85,0.91
4,KNN,"{'n_neighbors': 2, 'metric': 'euclidean'}",0.1197,3.1863,0.81,0.79,0.81,0.79,0.79
5,KNeighborsClassifier,"{'algorithm': 'auto', 'leaf_size': 30, 'metric...",0.0411,0.9278,0.83,0.82,0.83,0.82,0.86


# Ensemble

In [20]:
col_names = ["Model","Parameters","Training Time (sec)", "Testing Time (sec)", 
             "Accuracy","Precision", "Recall", "F1_score", "ROC-AUC"]
ensemble_results_table = pd.DataFrame(columns=col_names)

## Random Forest

In [21]:
# Select ensemble Model
ensemble_model = RandomForestClassifier(random_state=42)

# Set a list of parameters and # of folds
param_grid = {"n_estimators": [10, 50, 100],"max_depth":[None,2, 5,10]}
cv = 5

# Call the grid search function
grid_rf  = GridSearchCV(ensemble_model, param_grid=param_grid, cv=cv, scoring="accuracy")
grid_rf.fit(X_train, y_train)

results = pd.DataFrame(grid_rf.cv_results_)
results_sorted = results.sort_values(by="rank_test_score")

print(results_sorted[["params", "mean_test_score", "rank_test_score"]].head(5))

# Grab the best model
best_rf = grid_rf.best_estimator_

# Print the best parameters of the best estimator
print("Best Parameters:")
print(grid_rf.best_params_)

# Evaluate the model
row = evaluate_model(best_rf, X_train_std, X_test_std, y_train, y_test)

# Add metrics to results table 
ensemble_results_table = pd.concat([ensemble_results_table, row], ignore_index=True)

# print results table 
ensemble_results_table

                                      params  mean_test_score  rank_test_score
11    {'max_depth': 10, 'n_estimators': 100}         0.856112                1
10     {'max_depth': 10, 'n_estimators': 50}         0.854883                2
1    {'max_depth': None, 'n_estimators': 50}         0.851582                3
9      {'max_depth': 10, 'n_estimators': 10}         0.851313                4
2   {'max_depth': None, 'n_estimators': 100}         0.849969                5
Best Parameters:
{'max_depth': 10, 'n_estimators': 100}
Prediction time: 0.128012 seconds
Misclassifed examples: 904 out of 6513
Error: 0.139
Confusion_Matrix:
[[4743  202]
 [ 702  866]]
Accuracy: 0.861
Precision 0.857
Recall: 0.861
F1_score: 0.851
ROC_AUC score: 0.916


C:\Users\laura\AppData\Local\Temp\ipykernel_7112\290914579.py:28: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  ensemble_results_table = pd.concat([ensemble_results_table, row], ignore_index=True)


,Model,Parameters,Training Time (sec),Testing Time (sec),Accuracy,Precision,Recall,F1_score,ROC-AUC
0,RandomForestClassifier,"{'bootstrap': True, 'ccp_alpha': 0.0, 'class_w...",2.3353,0.128,0.86,0.86,0.86,0.85,0.92


## Majority Voting

In [27]:
# Select best classification model using gridsearch parameters from above
ensemble_model = "MajorityVoting"

estimators = [
    ("dt", DecisionTreeClassifier(criterion="gini", max_depth=10,random_state=42)),
    ("lr", LogisticRegression(C=1.0, penalty="l2",random_state=42)),
    ("knn", KNeighborsClassifier(n_neighbors=8, metric="euclidean"))
]

model, y_pred, row = ensemble_learning(ensemble_model, X_train_std, X_test_std, 
                  y_train, y_test,
                  estimators=estimators,voting="soft")

# Add metrics to results table 
ensemble_results_table = pd.concat([ensemble_results_table, row], ignore_index=True)

# print results table 
ensemble_results_table

Training time: 0.8413 seconds
Testing time: 0.9113 seconds
Misclassifed examples: 888 out of 6513
Error: 0.136
Confusion_Matrix:
[[4631  314]
 [ 574  994]]
Accuracy: 0.864
Precision 0.858
Recall: 0.864
F1_score: 0.859
ROC_AUC score: 0.915


,Model,Parameters,Training Time (sec),Testing Time (sec),Accuracy,Precision,Recall,F1_score,ROC-AUC
0,RandomForestClassifier,"{'bootstrap': True, 'ccp_alpha': 0.0, 'class_w...",2.3353,0.1280,0.86,0.86,0.86,0.85,0.92
1,MajorityVoting,"{'estimators': [('dt', DecisionTreeClassifier(...",0.8413,0.9113,0.86,0.86,0.86,0.86,0.91
